In [10]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import MarkdownHeaderTextSplitter
from langchain.text_splitter import CharacterTextSplitter
import os
import fitz  # pymupdf for PDF text extraction
import psycopg2
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings # or use SentenceTransformers
from langchain.vectorstores import Chroma , Pinecone
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
import numpy as np
import json

In [ ]:
import fitz  # PyMuPDF
import os
import json
from PIL import Image
import io

def find_caption_for_image(img_bbox, text_blocks, vertical_threshold=50):
    """
    Finds the most likely caption for an image based on proximity and keywords.
    """
    potential_captions = []
    for block in text_blocks:
        text_bbox = fitz.Rect(block[:4])
        block_text = block[4].strip()
        if block_text.lower().startswith(('figure', 'fig.', 'table')):
            vertical_distance = text_bbox.y0 - img_bbox.y1
            if -5 < vertical_distance < vertical_threshold:
                if img_bbox.x0 < text_bbox.x1 and img_bbox.x1 > text_bbox.x0:
                    potential_captions.append((vertical_distance, block_text))
    if not potential_captions:
        return None
    potential_captions.sort()
    best_caption = potential_captions[0][1].replace('\n', ' ').strip()
    return best_caption


def extract_images_and_save_captions(pdf_path, output_dir="output"):
    """
    Extracts images from a PDF, saves them, and stores their associated captions in a single JSON file.

    Args:
        pdf_path (str): The path to the PDF file.
        output_dir (str): The directory to save the extracted images and the final JSON file.
    """
    # Ensure the main output directory and the image subdirectory exist
    image_output_dir = os.path.join(output_dir, "images")
    if not os.path.exists(image_output_dir):
        os.makedirs(image_output_dir)

    doc = fitz.open(pdf_path)
    results = []
    
    print(f"Processing {len(doc)} pages in '{pdf_path}'...")

    for page_num in range(len(doc)):
        page = doc[page_num]
        text_blocks = page.get_text("blocks")
        images = page.get_images(full=True)
        
        for img_index, img_info in enumerate(images):
            xref = img_info[0]
            try:
                img_bbox = page.get_image_bbox(img_info)
            except ValueError:
                continue

            caption = find_caption_for_image(img_bbox, text_blocks)

            if caption:
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]
                image_ext = base_image["ext"]
                
                image_filename = f"page{page_num+1}_img{img_index+1}.{image_ext}"
                image_path = os.path.join(image_output_dir, image_filename)

                with open(image_path, "wb") as img_file:
                    img_file.write(image_bytes)

                results.append({
                    "image_path": image_path,
                    "caption": caption,
                    "source_page": page_num + 1
                })
                print(f"  [+] Found Image on Page {page_num+1}: '{caption[:50]}...' -> {image_path}")

    doc.close()

    # --- NEW: Save results to a JSON file ---
    json_output_path = os.path.join(output_dir, "image_captions.json")
    
    print(f"\nWriting {len(results)} extracted image-caption pairs to {json_output_path}...")
    
    # Use 'indent=4' for a human-readable, pretty-printed JSON file
    with open(json_output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=4, ensure_ascii=False)
        
    print("--- Extraction and JSON export complete! ---")
    return json_output_path


# --- Example Usage ---
if __name__ == "__main__":
    PDF_FILE = "../../data/raw/essentials-of-general-surgery-and-surgical-specialties_compress.pdf"  # Replace with your book's path
    OUTPUT_DIR = "../../data/imagescaption/"

    if not os.path.exists(PDF_FILE):
        print(f"Error: PDF file not found at '{PDF_FILE}'. Please update the path.")
    else:
        # The function now handles everything: extraction, saving images, and saving the JSON.
        final_json_path = extract_images_and_save_captions(PDF_FILE, OUTPUT_DIR)
        print(f"\nSuccess! All data saved. You can find your structured data in '{final_json_path}'")

Processing 1403 pages in '../../data/raw/essentials-of-general-surgery-and-surgical-specialties_compress.pdf'...
  [+] Found Image on Page 95: 'Figure 3-1 Nutrient absorption in the gastrointest...' -> ../../data/imagescaption/images\page95_img1.jpeg
  [+] Found Image on Page 109: 'Figure 4-1 The Lethal Triad....' -> ../../data/imagescaption/images\page109_img1.jpeg
  [+] Found Image on Page 134: 'Figure 6-2 A, The intra-aortic balloon pump is pla...' -> ../../data/imagescaption/images\page134_img1.jpeg
  [+] Found Image on Page 134: 'Figure 6-3 Bedside shock diagnosis and treatment. ...' -> ../../data/imagescaption/images\page134_img2.jpeg
  [+] Found Image on Page 137: 'Figure 6-4 Patients with chronic renal failure req...' -> ../../data/imagescaption/images\page137_img1.jpeg
  [+] Found Image on Page 138: 'Figure 6-5 Workup and management of acute renal fa...' -> ../../data/imagescaption/images\page138_img1.jpeg
  [+] Found Image on Page 150: 'Figure 7-1 Phases of wound healing comp

In [6]:
pdf_path = "../../data/raw/essentials-of-general-surgery-and-surgical-specialties_compress.pdf"

In [ ]:
loader = PyPDFLoader(pdf_path)
pages = loader.load()
print(f"Number of pages = {len(pages)}")

Number of pages = 1403


In [15]:
def extract_text_with_metadata(pdf_path, output_json="../../data/text_with_metadata.json"):
    doc = fitz.open(pdf_path)
    all_text = []

    for page_num, page in enumerate(doc):
        blocks = page.get_text("dict")["blocks"]
        
        for block in blocks:
            if "lines" in block:
                for line in block["lines"]:
                    for span in line["spans"]:
                        text_data = {
                            "page": page_num + 1,
                            "text": span["text"].strip(),
                            "font": span["font"],
                            "size": span["size"],
                            "color": span["color"],
                            "bbox": span["bbox"]  # (x0, y0, x1, y1)
                        }
                        # Ignore empty strings
                        if text_data["text"]:
                            all_text.append(text_data)

    # Save to JSON file
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(all_text, f, indent=2, ensure_ascii=False)

    print(f"✅ Extracted {len(all_text)} text spans with metadata.")
    return all_text

In [16]:
text = extract_text_with_metadata(pdf_path)

✅ Extracted 52116 text spans with metadata.


In [14]:
text

[{'page': 2,
  'text': 'Essentials of',
  'font': 'ACaslonPro-Bold',
  'size': 20.40280532836914,
  'color': 6107997,
  'bbox': (411.0289306640625,
   121.172119140625,
   523.27783203125,
   155.89662170410156)},
 {'page': 2,
  'text': 'General Surgery and',
  'font': 'ACaslonPro-Bold',
  'size': 20.40280532836914,
  'color': 6107997,
  'bbox': (343.2643737792969,
   145.1746826171875,
   523.27783203125,
   179.89918518066406)},
 {'page': 2,
  'text': 'Surgical Specialties',
  'font': 'ACaslonPro-Bold',
  'size': 20.40280532836914,
  'color': 6107997,
  'bbox': (356.6916198730469,
   169.17724609375,
   523.278564453125,
   203.90174865722656)},
 {'page': 2,
  'text': 'About the Cover:',
  'font': 'ACaslonPro-SemiboldItali',
  'size': 10.20140266418457,
  'color': 0,
  'bbox': (72.0, 288.6103515625, 138.46282958984375, 305.97259521484375)},
 {'page': 2,
  'text': 'Portrait of Dr. Samuel D. Gross (The Gross Clinic)',
  'font': 'ACaslonPro-Regular',
  'size': 10.20140266418457,
  'colo

In [ ]:
CHAPTER_FONT_SIZE = 20
TITLE_FONT_SIZE = 16
SUBTITLE_FONT_SIZE = 14

In [ ]:
def group_chunks_by_titles(spans):
    chunks = []
    current_chunk = {
        "chapter": None,
        "title": None,
        "subtitle": None,
        "content": []
    }

    for span in spans:
        text = span["text"]
        size = span["size"]
        font = span["font"]

        # Detect chapter
        if size >= 19 and "chapter" in text.lower():
            if current_chunk["content"]:
                chunks.append(current_chunk)
                current_chunk = {
                    "chapter": None,
                    "title": None,
                    "subtitle": None,
                    "content": []
                }
            current_chunk["chapter"] = text

        # Detect title
        elif size >= 16 and font.lower().endswith("bold"):
            if current_chunk["title"]:
                chunks.append(current_chunk)
                current_chunk = {
                    "chapter": current_chunk["chapter"],
                    "title": None,
                    "subtitle": None,
                    "content": []
                }
            current_chunk["title"] = text

        # Detect subtitle
        elif size >= 14:
            if current_chunk["subtitle"]:
                chunks.append(current_chunk)
                current_chunk = {
                    "chapter": current_chunk["chapter"],
                    "title": current_chunk["title"],
                    "subtitle": None,
                    "content": []
                }
            current_chunk["subtitle"] = text

        else:
            current_chunk["content"].append(text)

    # Append the last chunk
    if current_chunk["content"]:
        chunks.append(current_chunk)

    return chunks


In [3]:
page1 = pages [100]
print(page1.page_content)

Preexisting hemostatic defects should be suspected when a prior history of bleeding exists or when abnormal
bleeding begins within the first 30 minutes of the operative period. Bleeding disorders may be congenital
(
Table 4-2
) or acquired (
Table 4-3
).
TABLE 4-2
 Congenital Bleeding Disorders
Hemophilia A
Von Willebrand Disease
Incidence
25 per 100,000 in the United States
1% of U.S. population
Pathophysiology
Reduced or absent factor VIII activity.
Factor VIII molecule is present.
Reduced factor VIII activity and von Willebrand activity
Site of bleeding
Joints and intramuscular
Mucocutaneous
Inheritance
X-linked
Autosomal dominant
Patients
Only males
Males and females
Laboratory studies
Prolonged aPTT
Prolonged aPTT
Normal PT
Normal PT
 
Normal platelet function
Abnormal platelet function
aPTT, activated partial thromboplastin time; PT, prothrombin time.
TABLE 4-3
 Causes of Acquired Bleeding Disorders
Advanced liver disease
Anticoagulation therapy
Acquired thrombocytopenia
Platelet

In [4]:
print(page1.metadata)

{'producer': 'calibre 3.39.1 [https://calibre-ebook.com]', 'creator': 'calibre (3.39.1) [https://calibre-ebook.com]', 'creationdate': '2019-03-09T22:06:18+00:00', 'author': 'Peter F. Lawrence', 'moddate': '2019-03-10T06:10:18+08:00', 'title': 'Essentials of General Surgery and Surgical Specialties', 'source': '../../data/raw/essentials-of-general-surgery-and-surgical-specialties_compress.pdf', 'total_pages': 1403, 'page': 100, 'page_label': '101'}


In [5]:
# Character splitter 
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len
)

In [19]:
docs = text_splitter.split_documents(pages)

In [22]:
docs[300].page_content

'thromboembolism (VTE). The evaluation of patients for surgery must include an assessment of the degree of\nrisk the patient has for a VTE event. Virtually all surgery carries varying degrees of risk for VTE, from\nminimal to highly significant. A number of steps are essential in the assessment of the degree of risk in a\npatient.\nTABLE 4-5\n Differential Diagnosis of Hypercoagulable States by Site of Thrombosis\nArterial Thrombosis (e.g., Myocardial Infarction)\nVenous Thrombosis (e.g., VTE)\nCommon: Antiphospholipid syndrome\nCommon: Factor V Leiden\n    Prothrombin 20210 mutation\n    Prothrombin 20210\n    HIT Syndrome\n    Protein C deficiency\nUncommon: Elevated PAI-1 activity\nProtein S deficiency\n    Hyperhomocysteinemia (strokes in children)\n    Antithrombin deficiency\n    tPA Deficiency\n    Uncommon: Hyperhomocysteinemia (uremic patients)\nAnomalous coronary arteries\n    Factor XII deficiency\nVasculitis\nTrauma\nImmobilization'

In [2]:
pdf_folder_path = "../../data/raw/"  # Update with your folder path
output_folder = "../../data/images"  # Update with your output folder path

In [27]:
#Extract images from the PDF and save them to the specified output folder
def extract_images_from_pdf(pdf_path:str, output_folder:str)-> list:
    """Extract images from a PDF file and save them to the specified output folder.
    Args:
        pdf_path (str): Path to the PDF file.
        output_folder (str): Path to the output folder where images will be saved.  "
        "Returns:"
        "list: List of paths to the saved images.
    """
    doc = fitz.open(pdf_path)
    image_paths = []

    for page_num in range(len(doc)):
        for img_index, img in enumerate(doc[page_num].get_images(full=True)):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            img_filename = f"{output_folder}/page_{page_num + 1}_img_{img_index + 1}.png"

            with open(img_filename, "wb") as img_file:
                img_file.write(image_bytes)
            
            image_paths.append(img_filename)

    return image_paths  # Return list of image file paths

In [6]:
import fitz  # PyMuPDF
import os
import json

# Define paths
pdf_path = "../../data/raw/essentials-of-general-surgery-and-surgical-specialties_compress.pdf"
output_dir = "../../data/chunks/"
os.makedirs(output_dir, exist_ok=True)

# Heuristic: Identify headings by font size and color
def is_heading(span):
    try:
        return (
            span["size"] >= 16 and
            (span["color"] == 0 or span["color"] == 16711680)  # black or dark red
        )
    except KeyError:
        return False

def extract_chunks_by_titles(pdf_path):
    doc = fitz.open(pdf_path)
    chunks = []
    current_chunk = {
        "chapter": None,
        "section": None,
        "text": "",
        "page_start": None,
        "page_end": None,
        "chunk_id": None
    }

    for page_num, page in enumerate(doc, start=1):
        blocks = page.get_text("dict")["blocks"]
        for block in blocks:
            if "lines" not in block:
                continue
            for line in block["lines"]:
                line_text = " ".join([span["text"] for span in line["spans"] if span["text"].strip()])
                if not line_text:
                    continue

                for span in line["spans"]:
                    if is_heading(span):
                        if current_chunk["text"]:  # Save previous chunk
                            chunks.append(current_chunk)
                            current_chunk = {
                                "chapter": None,
                                "section": None,
                                "text": "",
                                "page_start": page_num,
                                "page_end": None,
                                "chunk_id": None
                            }

                        if line_text.upper().startswith("CHAPTER"):
                            current_chunk["chapter"] = line_text
                            current_chunk["section"] = None
                        else:
                            current_chunk["section"] = line_text
                            current_chunk["page_start"] = page_num
                        current_chunk["chunk_id"] = f"{current_chunk['chapter']}_{current_chunk['section']}_{page_num}".replace(" ", "_")
                    else:
                        current_chunk["text"] += line_text + "\n"

        current_chunk["page_end"] = page_num

    # Save the final chunk
    if current_chunk["text"]:
        chunks.append(current_chunk)

    # Save chunks to JSON files
    for i, chunk in enumerate(chunks):
        with open(os.path.join(output_dir, f"chunk_{i+1}.json"), "w", encoding="utf-8") as f:
            json.dump(chunk, f, indent=2, ensure_ascii=False)

    return len(chunks), output_dir

# Run extraction
num_chunks, chunk_folder = extract_chunks_by_titles(pdf_path)
num_chunks, chunk_folder


(36, '../../data/chunks/')

In [28]:
import pdfplumber
import pandas as pd

def extract_tables_from_pdf(pdf_path):
    tables_list = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables()
            for table in tables:
                df = pd.DataFrame(table)
                tables_list.append(df)
    
    return tables_list  # Return list of Pandas DataFrames

In [17]:
tables_list = extract_tables_from_pdf(pdf_folder_path)

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, def

In [21]:
tables_list[2]

,0
0,
1,C <3.0 >3.0 Severe Severe Poor >80


In [12]:
extract_images_from_pdf(pdf_folder_path, output_folder)

['../../data/images/page_1_img_1.png',
 '../../data/images/page_4_img_1.png',
 '../../data/images/page_60_img_1.png',
 '../../data/images/page_63_img_1.png',
 '../../data/images/page_65_img_1.png',
 '../../data/images/page_71_img_1.png',
 '../../data/images/page_95_img_1.png',
 '../../data/images/page_109_img_1.png',
 '../../data/images/page_132_img_1.png',
 '../../data/images/page_134_img_1.png',
 '../../data/images/page_134_img_2.png',
 '../../data/images/page_136_img_1.png',
 '../../data/images/page_136_img_2.png',
 '../../data/images/page_137_img_1.png',
 '../../data/images/page_138_img_1.png',
 '../../data/images/page_142_img_1.png',
 '../../data/images/page_150_img_1.png',
 '../../data/images/page_152_img_1.png',
 '../../data/images/page_153_img_1.png',
 '../../data/images/page_154_img_1.png',
 '../../data/images/page_173_img_1.png',
 '../../data/images/page_174_img_1.png',
 '../../data/images/page_180_img_1.png',
 '../../data/images/page_180_img_2.png',
 '../../data/images/page_

In [ ]:
"""from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model (you can use OpenAI, Cohere, etc.)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def semantic_chunking(text, chunk_size=500, chunk_overlap=100):
    """Splits text semantically based on embedding similarity."""
    # Split into sentences
    sentences = text.split(". ")
    embeddings = np.array([embedder.encode(s) for s in sentences])

    # Create chunks by grouping similar sentences
    chunks = []
    temp_chunk = []
    temp_embed = []
    
    for i, sentence in enumerate(sentences):
        temp_chunk.append(sentence)
        temp_embed.append(embeddings[i])

        if len(" ".join(temp_chunk)) > chunk_size:
            chunks.append(" ".join(temp_chunk))
            temp_chunk = temp_chunk[-(chunk_overlap // len(temp_chunk)):]  # Overlap
            temp_embed = temp_embed[-(chunk_overlap // len(temp_embed)):]
    
    if temp_chunk:
        chunks.append(" ".join(temp_chunk))
    
    return chunks
"""

In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
import re

class TitleBasedTextSplitter:
    def __init__(self, pattern=r"\n[A-Z\s]{3,}\n"):
        self.pattern = pattern

    def split_text(self, text):
        sections = re.split(self.pattern, text)
        titles = re.findall(self.pattern, text)
        chunks = []
        
        for i, section in enumerate(sections):
            if i < len(titles):
                title = titles[i].strip()
                chunks.append(title + "\n" + section.strip())
            else:
                chunks.append(section.strip())
        
        return chunks

In [13]:
import psycopg2

class PostgresManager:
    def __init__(self, db_name, user, password, host="localhost", port="5432"):
        try:
            self.conn = psycopg2.connect(
                dbname=db_name,
                user=user,
                password=password,
                host=host,
                port=port,
                options='-c client_encoding=UTF8'
            )
            self.cursor = self.conn.cursor()
            print("✅ Connected to PostgreSQL")
        except Exception as e:
            print("❌ Connection failed:", e)
            raise

    def create_table(self, table_name):
        query = f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            id SERIAL PRIMARY KEY,
            page_number INT,
            content TEXT
        );
        """
        self.cursor.execute(query)
        self.conn.commit()

    def insert_page(self, table_name, page_number, content):
        query = f"""
        INSERT INTO {table_name} (page_number, content)
        VALUES (%s, %s);
        """
        self.cursor.execute(query, (page_number, content))
        self.conn.commit()

    def fetch_all(self, table_name):
        self.cursor.execute(f"SELECT * FROM {table_name};")
        return self.cursor.fetchall()


    def reset_connection(self):
        self.conn.rollback()

    def close(self):
        self.cursor.close()
        self.conn.close()


In [14]:
manage_db = PostgresManager("rag_db","postgres","Jiji@jiji123")

✅ Connected to PostgreSQL


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter


client = QdrantClient("localhost", port=6333)  # Use cloud URL if needed

#Create a collection (if it doesn't exist)
collection_name = "mias_rag"
"""if collection_name not in client.get_collections().collections:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)  # 384 for MiniLM
    )
"""
def store_chunks_in_qdrant(chunks):
    embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    # Prepare documents
    payloads = []
    vectors = []
    
    for i, chunk in enumerate(chunks):
        vector = embeddings_model.embed_query(chunk)
        payloads.append({"chunk_id": i, "text": chunk})
        vectors.append(np.array(vector))

    # Insert into Qdrant
    client.upload_collection(
        collection_name=collection_name,
        vectors=vectors,
        payload=payloads,
    )
    
    print("✅ Chunks stored in Qdrant!")


embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# =============================
# 📌 1. Extract Text from PDFs
# =============================
def extract_text_from_pdf(pdf_path,table_name):
    doc = fitz.open(pdf_path)
    manage_db.create_table(table_name)
    page_number = 0
    for page in doc:
        content = page.get_text("text") + "\n"
        page_number +=1
        manage_db.insert_page(table_name,page_number,content)


def store_raw_text_in_postgres(text, filename):
    conn = psycopg2.connect(database="rag_db", user="jihed", password="Jiji@jiji123", host="localhost", port="5432")
    cur = conn.cursor()
    
    cur.execute("CREATE TABLE IF NOT EXISTS documents (id SERIAL PRIMARY KEY, filename TEXT, content TEXT)")

    # ✅ Convert text to UTF-8 and replace problematic characters
    text = text.encode("utf-8", "ignore").decode("utf-8")

    cur.execute("INSERT INTO documents (filename, content) VALUES (%s, %s)", (filename, text))
    
    conn.commit()
    cur.close()
    conn.close()



def chunk_text(text):
    """splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = splitter.split_text(text)"""

    splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
    chunks = splitter.split_text(text)# 🔹 Connect to Qdrant (Local or Cloud)
    return chunks


def store_chunks_in_chromadb(chunks):
    # Initialize ChromaDB and embeddings
    store_chunks_in_qdrant(chunks)
    

def process_pdfs(pdf_folder):
    manage_db.reset_connection()
    for pdf_file in os.listdir(pdf_folder):
        if pdf_file.endswith(".pdf"):
            pdf_path = os.path.join(pdf_folder, pdf_file)
            
            print(f"Processing {pdf_file}...")
            
            text = extract_text_from_pdf(pdf_path,"pdf_documents")
            #store_raw_text_in_postgres(text, pdf_file)
            #splitter = TitleBasedTextSplitter()
            #chunks = splitter.split_text(text)
            vector_db = store_chunks_in_qdrant(chunks)

    
    print("✅ Pipeline completed!")
    #return chunks

#chunks = process_pdfs(pdf_folder_path)


Processing 1853.PMC2518358.pdf...


NameError: name 'chunks' is not defined

In [30]:
chunks[4]

'SIXTH EDITION ENHANCEMENTS\nThis textbook was produced for medical students who are not planning a surgical career. We believe that all\nphysicians need to have a fundamental understanding of the options provided by surgery to be competent\nclinicians. Rather than using traditional textbook-writing techniques, the editors and authors of this textbook,\nmany of whom are members of the Association for Surgical Education, have conducted extensive research to\ndefine the content and skills needed for an optimal medical education program in surgery. Somewhat\nsurprisingly, there has been consensus among practicing surgeons and internists (and even psychiatrists!) about\nthe knowledge and skills in surgery needed by all physicians, and the information from this research has\nbecome the basis for this textbook.'

In [ ]:
"""client.create_collection(
    collection_name="mias_rag_2",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)"""

True

In [23]:
collections = client.get_collections()
print(collections)

collections=[CollectionDescription(name='mias_rag'), CollectionDescription(name='mias_rag_2')]
